In [6]:
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"

FEATURE_FILE = RESULTS_DIR / "training_features.csv"

df = pd.read_csv(FEATURE_FILE)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (1259849, 19)
Columns:
['name_exact', 'name_token_jaccard', 'name_char_similarity', 'name_token_set_ratio', 'name_token_sort_ratio', 'name_shared_tokens', 'name_length_ratio', 'address_exact', 'address_token_jaccard', 'address_char_similarity', 'address_token_set_ratio', 'address_token_sort_ratio', 'address_shared_tokens', 'address_length_ratio', 'country_match', 'name_missing', 'address_missing', 'label', 'source1_entity_id']


In [7]:
FEATURE_COLUMNS = [
    "name_exact",
    "name_token_jaccard",
    "name_char_similarity",
    "name_token_set_ratio",
    "name_token_sort_ratio",
    "name_shared_tokens",
    "name_length_ratio",

    "address_exact",
    "address_token_jaccard",
    "address_char_similarity",
    "address_token_set_ratio",
    "address_token_sort_ratio",
    "address_shared_tokens",
    "address_length_ratio",

    "country_match",
    "name_missing",
    "address_missing"
]

TARGET = "label"
GROUP = "source1_entity_id"

X = df[FEATURE_COLUMNS].astype(np.float32)
y = df[TARGET].astype(np.int8)
groups = df[GROUP]

print("X:", X.shape)
print("y:", y.shape)
print("Features:", len(FEATURE_COLUMNS))

X: (1259849, 17)
y: (1259849,)
Features: 17


In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

groups_train = groups.iloc[train_idx]
groups_val = groups.iloc[val_idx]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

print(
    "Train positive:",
    int(y_train.sum())
)

print(
    "Validation positive:",
    int(y_val.sum())
)

print(
    "S1 overlap:",
    len(
        set(groups_train)
        &
        set(groups_val)
    )
)

Train: (1011664, 17)
Validation: (248185, 17)
Train positive: 25309
Validation positive: 6403
S1 overlap: 0


In [9]:
from catboost import CatBoostClassifier

ModuleNotFoundError: No module named 'catboost'

In [ ]:
cat_model = CatBoostClassifier(
    iterations=500,
    depth=7,
    learning_rate=0.05,

    loss_function="Logloss",

    random_seed=42,

    verbose=100,

    thread_count=-1,

    allow_writing_files=False
)

cat_model.fit(
    X_train,
    y_train,

    eval_set=(X_val, y_val),

    use_best_model=True,

    verbose=100
)

In [ ]:
val_prob_cat = cat_model.predict_proba(
    X_val
)[:, 1]

print("Predictions:", len(val_prob_cat))
print("Min:", val_prob_cat.min())
print("Max:", val_prob_cat.max())
print("Mean:", val_prob_cat.mean())

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score
)

thresholds = np.arange(
    0.30,
    0.91,
    0.01
)

cat_threshold_results = []

for threshold in thresholds:

    pred = (
        val_prob_cat >= threshold
    ).astype(np.int8)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        pred,
        beta=0.5,
        zero_division=0
    )

    cat_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f0.5": f05
    })

cat_threshold_df = pd.DataFrame(
    cat_threshold_results
)

cat_threshold_df = cat_threshold_df.sort_values(
    "f0.5",
    ascending=False
)

cat_threshold_df.head(10)

In [ ]:
best_cat = cat_threshold_df.iloc[0]

print("========== CATBOOST BEST ==========")

print(
    f"Threshold: {best_cat['threshold']:.2f}"
)

print(
    f"Precision: {best_cat['precision']:.6f}"
)

print(
    f"Recall:    {best_cat['recall']:.6f}"
)

print(
    f"F1:        {best_cat['f1']:.6f}"
)

print(
    f"F0.5:      {best_cat['f0.5']:.6f}"
)

print("===================================")

In [ ]:
CAT_MODEL_PATH = RESULTS_DIR / "catboost_entity_resolution.cbm"

cat_model.save_model(
    CAT_MODEL_PATH
)

cat_threshold = float(
    best_cat["threshold"]
)

print("Model saved:", CAT_MODEL_PATH)
print("Best threshold:", cat_threshold)

In [11]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_X=X_val,
    eval_y=y_val
)

print("LightGBM training complete.")

LightGBM training complete.


In [12]:
val_prob_lgb = lgb_model.predict_proba(X_val)[:, 1]

print("Predictions:", len(val_prob_lgb))
print("Min:", val_prob_lgb.min())
print("Max:", val_prob_lgb.max())

Predictions: 248185
Min: 1.470583005864986e-08
Max: 0.9999792785699968


In [13]:
thresholds = np.arange(0.30, 0.91, 0.01)

lgb_threshold_results = []

for threshold in thresholds:

    pred = (val_prob_lgb >= threshold).astype(np.int8)

    precision = precision_score(
        y_val, pred, zero_division=0
    )

    recall = recall_score(
        y_val, pred, zero_division=0
    )

    f1 = f1_score(
        y_val, pred, zero_division=0
    )

    f05 = fbeta_score(
        y_val, pred,
        beta=0.5,
        zero_division=0
    )

    lgb_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f0.5": f05
    })

lgb_threshold_df = pd.DataFrame(
    lgb_threshold_results
).sort_values(
    "f0.5",
    ascending=False
)

lgb_threshold_df.head(10)

NameError: name 'precision_score' is not defined

In [ ]:
best_lgb = lgb_threshold_df.iloc[0]

print("========== LIGHTGBM BEST ==========")
print(f"Threshold: {best_lgb['threshold']:.2f}")
print(f"Precision: {best_lgb['precision']:.6f}")
print(f"Recall:    {best_lgb['recall']:.6f}")
print(f"F1:        {best_lgb['f1']:.6f}")
print(f"F0.5:      {best_lgb['f0.5']:.6f}")
print("====================================")

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

extra_model = ExtraTreesClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

extra_model.fit(X_train, y_train)

print("ExtraTrees training complete.")

In [ ]:
val_prob_extra = extra_model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
import numpy as np

thresholds = np.arange(0.30, 0.96, 0.01)

best_extra = None

for threshold in thresholds:
    val_pred = (val_prob_extra >= threshold).astype(int)

    precision = precision_score(y_val, val_pred, zero_division=0)
    recall = recall_score(y_val, val_pred, zero_division=0)
    f1 = f1_score(y_val, val_pred, zero_division=0)
    f05 = fbeta_score(y_val, val_pred, beta=0.5, zero_division=0)

    if best_extra is None or f05 > best_extra["f0.5"]:
        best_extra = {
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "f0.5": f05
        }

best_extra

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import fbeta_score
import numpy as np
import pandas as pd

configs = [
    # num_leaves, max_depth, min_child_samples
    (31,  6,  20),
    (63,  7,  20),
    (127, 8,  20),
    (63,  8,  50),
    (127, 10, 50),
    (63,  7, 100),
]

tuning_results = []

for leaves, depth, min_child in configs:

    print(
        f"Training: leaves={leaves}, "
        f"depth={depth}, "
        f"min_child={min_child}"
    )

    model = LGBMClassifier(
        n_estimators=500,
        max_depth=depth,
        num_leaves=leaves,
        learning_rate=0.05,
        min_child_samples=min_child,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_X=X_val,
        eval_y=y_val
    )

    probs = model.predict_proba(X_val)[:, 1]

    best_f05 = -1
    best_threshold = None
    best_precision = None
    best_recall = None

    for threshold in np.arange(0.50, 0.91, 0.01):

        preds = (probs >= threshold).astype(int)

        f05 = fbeta_score(
            y_val,
            preds,
            beta=0.5,
            zero_division=0
        )

        if f05 > best_f05:
            best_f05 = f05
            best_threshold = threshold

            from sklearn.metrics import precision_score, recall_score

            best_precision = precision_score(
                y_val, preds, zero_division=0
            )

            best_recall = recall_score(
                y_val, preds, zero_division=0
            )

    tuning_results.append({
        "num_leaves": leaves,
        "max_depth": depth,
        "min_child_samples": min_child,
        "threshold": best_threshold,
        "precision": best_precision,
        "recall": best_recall,
        "f0.5": best_f05
    })

results_df = pd.DataFrame(tuning_results)
results_df = results_df.sort_values("f0.5", ascending=False)

results_df

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import fbeta_score, precision_score, recall_score
import numpy as np
import pandas as pd

configs = [
    (96,  9, 50),
    (96, 10, 50),
    (127, 9, 50),
    (127, 10, 40),
    (127, 10, 75),
    (160, 10, 50),
]

tuning_results_2 = []
tuned_models = {}

for leaves, depth, min_child in configs:

    print(
        f"\nTraining: "
        f"leaves={leaves}, "
        f"depth={depth}, "
        f"min_child={min_child}"
    )

    model = LGBMClassifier(
        n_estimators=500,
        max_depth=depth,
        num_leaves=leaves,
        learning_rate=0.05,
        min_child_samples=min_child,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_X=X_val,
        eval_y=y_val
    )

    probs = model.predict_proba(X_val)[:, 1]

    best = None

    for threshold in np.arange(0.60, 0.81, 0.01):

        preds = (probs >= threshold).astype(int)

        precision = precision_score(
            y_val, preds, zero_division=0
        )

        recall = recall_score(
            y_val, preds, zero_division=0
        )

        f05 = fbeta_score(
            y_val,
            preds,
            beta=0.5,
            zero_division=0
        )

        if best is None or f05 > best["f0.5"]:
            best = {
                "threshold": threshold,
                "precision": precision,
                "recall": recall,
                "f0.5": f05
            }

    tuning_results_2.append({
        "num_leaves": leaves,
        "max_depth": depth,
        "min_child_samples": min_child,
        **best
    })

    tuned_models[
        (leaves, depth, min_child)
    ] = model

    print(
        f"Best threshold: {best['threshold']:.2f} | "
        f"Precision: {best['precision']:.6f} | "
        f"Recall: {best['recall']:.6f} | "
        f"F0.5: {best['f0.5']:.6f}"
    )


results_df_2 = pd.DataFrame(tuning_results_2)

results_df_2 = results_df_2.sort_values(
    "f0.5",
    ascending=False
).reset_index(drop=True)

results_df_2

In [ ]:
from lightgbm import LGBMClassifier
import joblib
from pathlib import Path

BEST_NUM_LEAVES = 127
BEST_MAX_DEPTH = 10
BEST_MIN_CHILD_SAMPLES = 50
BEST_THRESHOLD = 0.68

final_lgb_model = LGBMClassifier(
    n_estimators=500,
    max_depth=BEST_MAX_DEPTH,
    num_leaves=BEST_NUM_LEAVES,
    learning_rate=0.05,
    min_child_samples=BEST_MIN_CHILD_SAMPLES,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

final_lgb_model.fit(
    X,
    y
)

print("Final LightGBM training complete.")

In [ ]:
RESULTS_DIR = Path("../results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = RESULTS_DIR / "lightgbm_entity_resolution.txt"

final_lgb_model.booster_.save_model(str(model_path))

threshold_path = RESULTS_DIR / "lightgbm_decision_threshold.txt"

threshold_path.write_text(
    str(BEST_THRESHOLD),
    encoding="utf-8"
)

print("Model saved:", model_path)
print("Threshold saved:", threshold_path)

In [ ]:
print("Model exists:", model_path.exists())
print("Threshold exists:", threshold_path.exists())
print("Threshold:", threshold_path.read_text())

In [14]:
import pandas as pd
import numpy as np
import xgboost as xgb

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, fbeta_score

RESULTS_DIR = Path("../results").resolve()

# Load the exact training features
data = pd.read_csv(RESULTS_DIR / "training_features.csv")

FEATURE_COLUMNS = [
    "name_exact",
    "name_token_jaccard",
    "name_char_similarity",
    "name_token_set_ratio",
    "name_token_sort_ratio",
    "name_shared_tokens",
    "name_length_ratio",
    "address_exact",
    "address_token_jaccard",
    "address_char_similarity",
    "address_token_set_ratio",
    "address_token_sort_ratio",
    "address_shared_tokens",
    "address_length_ratio",
    "country_match",
    "name_missing",
    "address_missing"
]

X = data[FEATURE_COLUMNS]
y = data["label"]
groups = data["source1_entity_id"]

# EXACT same group split used previously
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X, y, groups=groups)
)

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

print("Validation shape:", X_val.shape)
print("Validation positives:", y_val.sum())
print("Validation negatives:", (y_val == 0).sum())

Validation shape: (248185, 17)
Validation positives: 6403
Validation negatives: 241782


In [15]:
# Load previously trained XGBoost model
xgb_model = xgb.XGBClassifier()

xgb_model.load_model(
    RESULTS_DIR / "xgboost_entity_resolution.json"
)

print("XGBoost loaded.")

XGBoost loaded.


In [16]:
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

print("XGB validation probabilities generated.")
print("Min:", xgb_val_prob.min())
print("Max:", xgb_val_prob.max())
print("Unique:", len(np.unique(xgb_val_prob)))

XGB validation probabilities generated.
Min: 3.160822e-07
Max: 0.9997961
Unique: 227061


In [18]:
# Compare XGB + LightGBM decision rules on VALIDATION data

rules = {
    "XGB_only": (
        (xgb_val_prob >= 0.67)
    ),

    "LGB_only": (
        (val_prob_lgb >= 0.73)
    ),

    "BOTH": (
        (xgb_val_prob >= 0.67) &
        (val_prob_lgb >= 0.73)
    ),

    "EITHER": (
        (xgb_val_prob >= 0.67) |
        (val_prob_lgb >= 0.73)
    ),

    "XGB_high_or_both": (
        (xgb_val_prob >= 0.73) |
        (
            (xgb_val_prob >= 0.67) &
            (val_prob_lgb >= 0.73)
        )
    )
}

for name, pred in rules.items():
    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    f05 = fbeta_score(y_val, pred, beta=0.5, zero_division=0)

    print(
        f"{name:22s} "
        f"Precision={precision:.5f} "
        f"Recall={recall:.5f} "
        f"F0.5={f05:.5f}"
    )

XGB_only               Precision=0.97109 Recall=0.84976 F0.5=0.94413
LGB_only               Precision=0.96282 Recall=0.81290 F0.5=0.92857
BOTH                   Precision=0.97507 Recall=0.80618 F0.5=0.93586
EITHER                 Precision=0.95958 Recall=0.85647 F0.5=0.93702
XGB_high_or_both       Precision=0.97435 Recall=0.83070 F0.5=0.94178


In [19]:
import xgboost as xgb

xgb_val_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_val_model.fit(
    X_train,
    y_train
)

print("Fresh XGBoost validation model trained.")

Fresh XGBoost validation model trained.


In [20]:
xgb_val_prob = xgb_val_model.predict_proba(X_val)[:, 1]

print("XGB validation probabilities ready.")
print("Min:", xgb_val_prob.min())
print("Max:", xgb_val_prob.max())
print("Unique:", len(np.unique(xgb_val_prob)))

XGB validation probabilities ready.
Min: 2.9397577e-07
Max: 0.99990225
Unique: 228512


In [21]:
pred_xgb = xgb_val_prob >= 0.67

print(
    "XGB @ 0.67:",
    "Precision =", precision_score(y_val, pred_xgb),
    "Recall =", recall_score(y_val, pred_xgb),
    "F0.5 =", fbeta_score(y_val, pred_xgb, beta=0.5)
)

XGB @ 0.67: Precision = 0.9555394903307428 Recall = 0.8257066999843823 F0.5 = 0.9264061678640266


In [22]:
rules = {
    "XGB_only": (
        xgb_val_prob >= 0.67
    ),

    "LGB_only": (
        val_prob_lgb >= 0.73
    ),

    "BOTH": (
        (xgb_val_prob >= 0.67) &
        (val_prob_lgb >= 0.73)
    ),

    "EITHER": (
        (xgb_val_prob >= 0.67) |
        (val_prob_lgb >= 0.73)
    ),

    "XGB_high_or_both": (
        (xgb_val_prob >= 0.73) |
        (
            (xgb_val_prob >= 0.67) &
            (val_prob_lgb >= 0.73)
        )
    )
}

for name, pred in rules.items():

    precision = precision_score(
        y_val, pred, zero_division=0
    )

    recall = recall_score(
        y_val, pred, zero_division=0
    )

    f05 = fbeta_score(
        y_val, pred,
        beta=0.5,
        zero_division=0
    )

    print(
        f"{name:22s} "
        f"Precision={precision:.5f} "
        f"Recall={recall:.5f} "
        f"F0.5={f05:.5f}"
    )

XGB_only               Precision=0.95554 Recall=0.82571 F0.5=0.92641
LGB_only               Precision=0.96282 Recall=0.81290 F0.5=0.92857
BOTH                   Precision=0.96748 Recall=0.79916 F0.5=0.92837
EITHER                 Precision=0.95133 Recall=0.83945 F0.5=0.92663
XGB_high_or_both       Precision=0.96296 Recall=0.81212 F0.5=0.92847


In [24]:
thresholds = np.arange(0.60, 0.81, 0.01)

best = None

for t in thresholds:
    pred = val_prob_lgb >= t

    precision = precision_score(
        y_val, pred, zero_division=0
    )
    recall = recall_score(
        y_val, pred, zero_division=0
    )
    f05 = fbeta_score(
        y_val, pred,
        beta=0.5,
        zero_division=0
    )

    if best is None or f05 > best["F0.5"]:
        best = {
            "threshold": t,
            "precision": precision,
            "recall": recall,
            "F0.5": f05
        }

print("Best LightGBM threshold:")
print(best)

Best LightGBM threshold:
{'threshold': np.float64(0.7500000000000001), 'precision': 0.9662035298535486, 'recall': 0.8036857722942371, 'F0.5': 0.9286461904933772}


In [25]:
# Fine threshold search for XGB + LightGBM

xgb_thresholds = np.arange(0.62, 0.76, 0.01)
lgb_thresholds = np.arange(0.70, 0.81, 0.01)

best_hybrid = None

for xt in xgb_thresholds:
    for lt in lgb_thresholds:

        # Keep if BOTH models consider it a match
        pred = (
            (xgb_val_prob >= xt) &
            (val_prob_lgb >= lt)
        )

        f05 = fbeta_score(
            y_val,
            pred,
            beta=0.5,
            zero_division=0
        )

        if best_hybrid is None or f05 > best_hybrid["F0.5"]:
            best_hybrid = {
                "xgb_threshold": xt,
                "lgb_threshold": lt,
                "precision": precision_score(
                    y_val, pred, zero_division=0
                ),
                "recall": recall_score(
                    y_val, pred, zero_division=0
                ),
                "F0.5": f05
            }

print("Best BOTH-model rule:")
print(best_hybrid)

Best BOTH-model rule:
{'xgb_threshold': np.float64(0.62), 'lgb_threshold': np.float64(0.72), 'precision': 0.9652401420295272, 'recall': 0.8066531313446822, 'F0.5': 0.9287229834214407}


In [26]:
best_either = None

for xt in xgb_thresholds:
    for lt in lgb_thresholds:

        pred = (
            (xgb_val_prob >= xt) |
            (val_prob_lgb >= lt)
        )

        f05 = fbeta_score(
            y_val,
            pred,
            beta=0.5,
            zero_division=0
        )

        if best_either is None or f05 > best_either["F0.5"]:
            best_either = {
                "xgb_threshold": xt,
                "lgb_threshold": lt,
                "precision": precision_score(
                    y_val, pred, zero_division=0
                ),
                "recall": recall_score(
                    y_val, pred, zero_division=0
                ),
                "F0.5": f05
            }

print("Best EITHER-model rule:")
print(best_either)

Best EITHER-model rule:
{'xgb_threshold': np.float64(0.7500000000000001), 'lgb_threshold': np.float64(0.75), 'precision': 0.962451684152402, 'recall': 0.8166484460409184, 'F0.5': 0.929269593033588}


In [27]:
print("Best EITHER rule:")
print(best_either)

print("\nUse:")
print("XGB threshold =", best_either["xgb_threshold"])
print("LGB threshold =", best_either["lgb_threshold"])

Best EITHER rule:
{'xgb_threshold': np.float64(0.7500000000000001), 'lgb_threshold': np.float64(0.75), 'precision': 0.962451684152402, 'recall': 0.8166484460409184, 'F0.5': 0.929269593033588}

Use:
XGB threshold = 0.7500000000000001
LGB threshold = 0.75
